# 2.4 – Advanced Feature Engineering Methods

A typical approach in Data Science is "featurization of the Universe". This means we could extract and engineer all the features possible for a given problem.

To give an example: In a time series problem, one could use cumulative sums, moving averages with variable window sizes, discrete state changes, average differences, etc. as features, which quickly becomes very large. Alternatively, data source can be so rich that already the raw extraction produces a vast amount of data.

In this case exploratory data analysis is challenging and we need to resort to alternative methods of visualizing and exploring the feature space.

One approach is to reduce the dimensionality of the feature space and poke around in this reduced feature space. A basic technique well-suited for this problem is the Principal Component Analysis which tries to find the directions of most variation in your data set. 

## Principal Component Analysis (PCA)
Visualizing 2 or 3-dimensional data is not that challenging. However, even the Iris dataset is 4-dimensional. We can use PCA to reduce that 4-dimensional data into 2 or 3 dimensions so that you can plot and hopefully understand the data better.

Reducing the size of the feature space sounds helpful, but might cause different problems. Therefore, always beware of two key challenges for dimensionality reduction:
1. We want to **retain as much of the information as possible**. If we blindly remove information, the world becomes easier, but we cannot apply machine learning anymore.
2. We want to **keep the information interpretable**. If we randomly convolute information, it might yield alright results, but as humans we cannot make sense of these results anymore.

The Iris dataset consists of 50 samples from each of three species of Iris (Iris setosa, Iris virginica and Iris versicolor). Four features were measured from each sample: the length and the width of the sepals and petals, in centimeters.

![](./data/3_2_iris.png)

### Data Preparation

In [ ]:
from pathlib import Path

candidate_paths = [
  parent / "data" / "2_4_iris.csv"
  for parent in (Path.cwd(), *Path.cwd().parents)
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
  raise FileNotFoundError("Could not find data/2_4_iris.csv in the repository.")

print(f"Loading: {data_path.resolve()}")

In [ ]:
import pandas as pd

df = pd.read_csv(data_path)

df.head()

#### **TASK: Data Prepration**
PCA is effected by scale so you need to scale the features in your data before applying PCA. 

Use StandardScaler to help you standardize the dataset’s features (mean = 0 and variance = 1) and label the resulting dataframe `x_scaled`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Code up your solution here...

### Dimensionality Reduction
The original data has 4 columns (sepal length, sepal width, petal length, and petal width). We want to project the original data which is 4-dimensional into 2 dimensions. After dimensionality reduction, there usually isn’t a particular meaning assigned to each principal component. The new components are just the two main dimensions of variation.

To that, we apply a [PCA](http://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) for two dimensions ("components") on the standardized data set.

In [ ]:
from sklearn.decomposition import PCA

# select parameters for PCA: 2 components, fixed random state for reproducibility
pca = PCA(n_components=2, random_state=42)
# fit and transform the data, i.e., project it onto the 2 principal components
principalComponents = pca.fit_transform(x_scaled)
# create a DataFrame with the principal components
df_pc = pd.DataFrame(data = principalComponents, columns = ['PC 1', 'PC 2'])

Now we are able to plot the two principal components:

In [ ]:
df_pc['target'] = df['target']
df_pc.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# map target names to colors
colors = {'Iris-setosa': 'red', 'Iris-versicolor': 'green', 'Iris-virginica': 'blue'}
df_pc['target2'] = df_pc['target'].map(colors)

# create a scatter plot of the two principal components
plt.scatter(df_pc['PC 1'], df_pc['PC 2'], c=df_pc['target2'], s=50)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Iris species')

# create legend items
patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
plt.legend(handles=patches)
plt.show()

#### **TASK: Dimensionality Reduction to 2D**

How would you describe the plot?
Can you interpret the principal components?

*Description*

...

*Interpretation*

...

### Explained Variance and Loadings

The **explained variance** tells you how much information (variance) can be attributed to each of the principal components. This is important because you lose some of the variance (information) when you convert a 4-dimensional space to a 2-dimensional space. By using the attribute `explained_variance_ratio_`, you can see that the first principal component contains 72.77% of the variance and the second principal component contains 23.03% of the variance. Together, the two components contain 95.80% of the information.

In [ ]:
# explained variance ratio for 2D PCA
pca.explained_variance_ratio_

In [ ]:
explained_variance = pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]
# explained_variance in percent
print("Total variance explained: " + str(explained_variance.round(4)*100) + "%")

**Loadings** show how strongly each original feature contributes to a principal component. Mathematically speaking, they are the coefficients of the linear combinations that define the PCs.
- absolute loading = influence of feature on PC
- positive or negative value = direction of that relationship

In [ ]:
# get feature names from all dataset columns except 'target'
feature_names = df.columns[:-1]  

# loadings are the contribution of each feature to each PC, shape: (n_features, n_components)
loadings = pca.components_.T

# plot loadings
x_loadings, y_loadings = loadings[:, 0], loadings[:, 1]
plt.scatter(x_loadings, y_loadings, color='orange', s=80)
for i, name in enumerate(feature_names):
  plt.text(x_loadings[i]*1.1, y_loadings[i]*1.1, name, ha='center', va='center')
plt.title("Feature Loadings in 2D")
plt.xlabel("PC1 loading")
plt.ylabel("PC2 loading")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

The loading plot visualizes the contribution of each original feature to the principal components PC1 and PC2. Each point represents one feature.
- `sepal width` seems to be pretty similar to the original feature because it has almost a loading of 1 for PC2.
- `petal width` and `petal length` are combined into PC1 and share similar loadings.
- `sepal length` is somewhere in between and slightly loads onto both, PC1 and PC2.

Together with the species plot, we can say that `petal width`, `petal length`, and `sepal length` separate `Iris-setosa` from the other species. However, all species have similar distributions of `sepal width`.

#### **TASK: Explained Variance and Interpretation**

1. Compute a principal component analysis for one dimension
2. Try to interpret the single principal component

In [ ]:
# Code up your solution here...

In [ ]:
# Code up your solution here...